# Custom Exploratory Data Analysis & Feature Profiling

This notebook conducts exploratory analysis on the retail loan dataset to understand key risk drivers, identify outliers, and define preprocessing transformations before model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid")
raw_csv_path = r"C:\Users\siddp\Downloads\Dataset for default loan prediction\loan.csv"

## 1. Load Data Sample
Since the full dataset is 1.18 GB (~2.26 million rows), we load a random sample of 100,000 rows for analysis.

In [ ]:
print("Loading a sample of 100,000 rows from the dataset...")
df_sample = pd.read_csv(raw_csv_path, nrows=100000)
print(f"Data shape loaded: {df_sample.shape}")

## 2. Target Variable Analysis
Inspect the `loan_status` values to decide how to construct our binary target `is_default`.

In [ ]:
status_counts = df_sample['loan_status'].value_counts()
print("Loan Status distribution:")
print(status_counts)

# Define binary default target
default_categories = [
    'Charged Off', 
    'Default', 
    'Does not meet the credit policy. Status:Charged Off', 
    'Late (31-120 days)'
]
df_sample['is_default'] = df_sample['loan_status'].isin(default_categories).astype(int)

print("\nBinary Target (is_default) Balance:")
print(df_sample['is_default'].value_counts(normalize=True))

## 3. Key Predictors Profiling
Check distributions and potential outliers for key loan fields: `loan_amnt`, `annual_inc`, and `dti`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loan Amount Distribution
sns.histplot(df_sample['loan_amnt'], bins=30, ax=axes[0], kde=True, color='blue')
axes[0].set_title('Loan Amount Distribution')

# Annual Income Distribution (log scale to handle high skewness)
sns.histplot(df_sample['annual_inc'].dropna(), bins=30, ax=axes[1], kde=True, color='green')
axes[1].set_yscale('log')
axes[1].set_title('Annual Income Distribution (Log Scale)')

# Debt-to-Income (DTI)
sns.histplot(df_sample['dti'].dropna(), bins=30, ax=axes[2], kde=True, color='red')
axes[2].set_title('Debt-to-Income (DTI) Distribution')

plt.tight_layout()
plt.show()

## 4. Bivariate Analysis: Risk Rates by Grade
Analyze default rates across grades to verify grade classification signal.

In [ ]:
default_by_grade = df_sample.groupby('grade')['is_default'].mean().reset_index()
default_by_grade['default_rate_pct'] = default_by_grade['is_default'] * 100

plt.figure(figsize=(10, 5))
sns.barplot(data=default_by_grade.sort_values('grade'), x='grade', y='default_rate_pct', palette='viridis')
plt.title('Default Rate (%) by Loan Grade')
plt.ylabel('Default Rate (%)')
plt.xlabel('Loan Grade')
plt.show()